# Measuring tune response ophyd async style

## import and set up

###  standard modules

In [1]:
import logging


import yaml
import jsons
import pprint

additional modules for extra info

In [2]:
import asyncio
import itertools
import json
import os
import pprint
import yaml

### logging level

In [3]:
# avoid being messed up with communication debug messages
logging.basicConfig(level=logging.WARNING)


### accml and accml lib modules

In [4]:
from accml.core.utils.basic_measurement_execution_engine import BasicMeasurementExecutionEngine
from accml.core.utils.simple_storage import SimpleDataStorage
from accml.custom.ophyd_async.ophyd_async_backend import OphydAsyncDeviceBackendRW
from accml.custom.ophyd_async.ophyd_async_delta_backend import OphydAsyncDeltaBackendRWProxy
from accml_lib.core.bl.command_rewritter import CommandRewriter
from accml_lib.core.bl.delta_backend import StateCache
from accml_lib.core.model.output.tune import Tune
from accml_lib.custom.bessyii.liasion_translator_setup import load_managers

from accml.app.tune.model import TuneResponseCollection

In [5]:
from accml_lib.core.model.utils.command import Command, CommandSequence, ReadCommand, TransactionCommand, BehaviourOnError

### jsons for exporting data model

jsons takes care to export the datamodel into basic types (lists, dictonaries, int, string, float...)
the output can then be stored in appropriate formates eg. json or yaml

In [6]:
from accml_lib.core.model import jsons_support

jsons_fork = jsons.fork()
jsons_support.register_serializers(jsons_fork)

## device setup and connection

### connection to devices

In [7]:
from accml_lib.custom.bessyii.setup import setup

/home/mfp/workspace/accml-main-lib-splitup/accml/venv-accml-tst/lib/python3.10/site-packages/wheel/bdist_wheel.py:4: FutureWarning: The 'wheel' package is no longer the canonical location of the 'bdist_wheel' command, and will be removed in a future release. Please update to setuptools v70.1 or later which contains an integrated version of this command.
  warn(


For the machine: typically no additional prefix

In [8]:
devices = setup(prefix="")

for the twin:

  * give the prefix, don't forget to add a ":" if you use the Twin's defaukt
  * set it to None: then it will use its external mimic and evaluate the OS

In [9]:
devices = setup(prefix=os.getenv("USER") +":")

In [10]:
await devices.get("tune").connect()


Todo:
    Need to fix emil quadrupole in BESSYII twin

In [11]:
await asyncio.gather( *[pc.connect() for name, pc in devices.get('quadrupole_pcs').settable_devices.items() if "QI" not in pc.name]);

### Manager setup

In [12]:
yp, lm, ts = load_managers()

In [13]:
pprint.pprint(yp)

In [14]:
pprint.pprint(lm)

In [15]:
pprint.pprint(ts)

### setup of measurement execution engine

In [16]:
backend = OphydAsyncDeviceBackendRW(devices=devices)

In [17]:
state_cache=StateCache(name="BESSY2_OphydAsync_Dev_State_Cache")
delta_backend = OphydAsyncDeltaBackendRWProxy(
    backend=backend,
    cache=state_cache,
)

In [18]:
storage = SimpleDataStorage()

In [19]:
mexec = BasicMeasurementExecutionEngine(
    backend=delta_backend,
    cmd_rewriter=CommandRewriter(liaison_manager=lm, translation_service=ts),
    storage=storage,
    expected_view_for_output="device",
    num_readings=2
)


## The measurement itself

### The commands

In [20]:
measurement_values = [0, -1, 0, 1, 0]

In [21]:
cmds_on_machine = []
for name in yp.get("quadrupole_pcs"):
    for val in measurement_values:
        tc = TransactionCommand(
                transaction=[Command(id=name, property="delta_set_current", value=val, behaviour_on_error=BehaviourOnError.stop)]
            )
        # need to execute it more than once: working on real twin
        cmds_on_machine.extend([tc] * 3)
cmds_on_machine = CommandSequence(commands=cmds_on_machine)


In [22]:
from typing import Sequence

def device_identifers_in_commands(commands: Sequence[TransactionCommand]) -> Sequence[str]:
    return tuple(
        itertools.chain(*[
            [cmd.id for cmd in tc.transaction]
            for tc in cmds_on_machine.commands
        ])
    )

In [23]:
device_identifers_in_commands(cmds_on_machine);

In [24]:
def contains(tc: TransactionCommand, name: str) -> bool:
    for cmd in tc.transaction:
        if cmd.id == name:
            return True
    return False

sel = CommandSequence(
    commands=list(
        filter(lambda tc: contains(tc, "Q3PD1R"), cmds_on_machine.commands)
    )
)
pprint.pprint(sel)

CommandSequence(commands=[TransactionCommand(transaction=[Command(id='Q3PD1R',
                                                                  property='delta_set_current',
                                                                  value=0,
                                                                  behaviour_on_error=<BehaviourOnError.stop: 1>)]),
                          TransactionCommand(transaction=[Command(id='Q3PD1R',
                                                                  property='delta_set_current',
                                                                  value=0,
                                                                  behaviour_on_error=<BehaviourOnError.stop: 1>)]),
                          TransactionCommand(transaction=[Command(id='Q3PD1R',
                                                                  property='delta_set_current',
                                                                  value=0,
                  

I want to be sure that I exclude one power converter (the emil quadrupole to be precise)

In [25]:
def not_contains(tc: TransactionCommand, name: str) -> bool:
    for cmd in tc.transaction:
        if cmd.id != name:
            return True
    return False

[name for name in yp.get("quadrupoles") if "QI" in name]    

['QIT6R']

In [26]:
cmds_on_machine = CommandSequence(
    commands=list(
        filter(lambda tc: not_contains(tc, "QIT6R"), cmds_on_machine.commands)
    )
)
# pprint.pprint(sel)

In [27]:
assert "Q1T6R" not  in  itertools.chain(*[
        [cmd.id for cmd in tc.transaction]
        for tc in cmds_on_machine.commands
    ])

## Taking data: a single value

### using backend directly

In [28]:
await backend.trigger(dev_id="Q3PD1R", prop_id="set_current")

In [29]:
r = await backend.read(dev_id="Q3PD1R", prop_id="set_current")
r

{'Q3PD1R-set_current': {'value': 181.1700988720213,
  'timestamp': 1769855912.485533,
  'alarm_severity': 0}}

In [30]:
backend.devices._devices.keys()

dict_keys(['quadrupole_pcs', 'master_clock', 'tune', 'steerer_pcs', 'orbit', 'Q3PT5R', 'Q3PD4R', 'Q4P1T1R', 'Q4PD1R', 'Q1PTR', 'Q3PT3R', 'Q4PD3R', 'Q3PD1R', 'Q3PD8R', 'Q3PD2R', 'Q3PT4R', 'Q4PD8R', 'Q3P1T1R', 'Q4PD5R', 'Q4PD2R', 'Q5P2T8R', 'Q3P1T8R', 'Q5PT5R', 'Q3PD3R', 'Q3PT7R', 'Q2PTR', 'Q3PD6R', 'Q4PT4R', 'Q4P1T6R', 'Q4PT5R', 'Q5P2T6R', 'Q5PT3R', 'Q4PT7R', 'Q5P1T6R', 'Q5PT4R', 'Q4P2T1R', 'Q5P1T1R', 'Q4PT3R', 'Q5PT2R', 'Q3PT2R', 'Q4PD4R', 'Q2PDR', 'Q4PT2R', 'Q5PT7R', 'Q1PDR', 'Q3P2T6R', 'Q4PD7R', 'Q4P2T6R', 'Q3PD7R', 'Q3PD5R', 'Q4P2T8R', 'Q5P1T8R', 'Q5P2T1R', 'Q3P1T6R', 'Q3P2T8R', 'Q4PD6R', 'PQIT6R', 'Q3P2T1R', 'Q4P1T8R', 'mc-frequency', 'VS2P1T2R', 'VS3P1T2R', 'VS4P1T2R', 'VS4P2T2R'])

### using measurement execution engine

In [31]:
await mexec.trigger_read([ReadCommand(id="Q3PD1R", property="set_current")])

ReadTogether(data=[SingleReading(name='Q3PD1R-set_current', payload={'Q3PD1R-set_current': {'value': 181.1700988720213, 'timestamp': 1769855912.485533, 'alarm_severity': 0}}, cmd=ReadCommand(id='Q3PD1R', property='set_current'))], start=datetime.datetime(2026, 1, 31, 12, 31, 21, 238987), end=datetime.datetime(2026, 1, 31, 12, 31, 21, 240912))

### now lets get under the hood: ophyd async

In [32]:
backend.get_natural_view_name()

'device'

In [33]:
dev = backend.devices.get("Q3PD1R")

In [34]:
# await dev.trigger()

In [35]:
await dev.read()

{'Q3PD1R-units': {'value': '',
  'timestamp': 1769855912.485495,
  'alarm_severity': 0},
 'Q3PD1R-set_current': {'value': 181.1700988720213,
  'timestamp': 1769855912.485533,
  'alarm_severity': 0},
 'Q3PD1R-precision': {'value': 2,
  'timestamp': 1769855912.485495,
  'alarm_severity': 0},
 'Q3PD1R-readback': {'value': 181.1700988720213,
  'timestamp': 1769855912.485495,
  'alarm_severity': 0}}

### Take full  measurement

In [36]:
!pwd

/home/mfp/workspace/accml-main-lib-splitup/accml/examples/notebooks/tune


In [37]:
detectors = [ReadCommand(id="tune", property="transversal")]

In [38]:
md = {}

uid = await mexec.execute(
    detectors=detectors,
    commands_collection=sel.commands,  # need to add bpms
    md=md,
)

#uid = await mexec.execute(
#    detectors=detectors,
#    commands_collection=cmds_on_machine.commands,  # need to add bpms
#    md=md,
#)


uid,

100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 15/15 [00:29<00:00,  1.94s/it]


(UUID('e503b30c-f3d4-4d25-ac8c-452f1280dd7e'),)

### Analysis preparation: what happened  

#### reference point: inspect cache

In [39]:
for k, v in state_cache.cache.items():
    data = v[f"{k.id}-set_current"]
    print(f'{k.id:8s}: {data["value"]}')

Q3PD1R  : 181.1700988720213


In [40]:
state_cache.get(ReadCommand(id="Q4PT2R", property="set_current"))

In [41]:
data = storage.get(uid)
data = jsons.dump(data, fork_inst=jsons_fork)

with open("tune_response_data_from_simulator.json", "wt") as fp:
    json.dump(data, fp, indent=4)

/home/mfp/workspace/accml-main-lib-splitup/accml/venv-accml-tst/lib/python3.10/site-packages/jsons/_common_impl.py:43: UserWarning: The use of datetimes without timezone is dangerous and can lead to undesired results. Use suppress_warning(datetime-without-tz) or suppress_warnings(True) to turn off this message.
  warnings.warn(msg_, *args, **kwargs)


In [42]:
with open("tune_response_data_from_simulator.yml", "wt") as fp:
    yaml.dump(data, fp, indent=4)